# Answer + measure v0: BM25 top-3 pages → Qwen3-VL-4B → judged

For every **dev** question:

1. BM25 finds the top 3 pages (same as notebook 03).
2. **Answer:** the page *images* go to Qwen3-VL-4B (served by vLLM), which returns JSON:
   `{"answer": ..., "citations": [page ids], "abstain": true/false}`.
3. **Measure:** a judge model compares each answer with the reference answer; citations are checked against the correct pages.

**Before running:** the vLLM server must be up on this GPU node (see the reply in chat for the command).

## 1. Settings, data, BM25 top-3

In [1]:
import base64
import io
import json
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import bm25s
from datasets import load_dataset
from openai import OpenAI
from tqdm.auto import tqdm

BASE_URL = "http://127.0.0.1:8000/v1"           # the vLLM server on this node
MODEL = "Qwen/Qwen3-VL-4B-Instruct"
TOP_PAGES = 3                                   # pages shown to the model per question
MAX_SIDE = 1600                                 # resize page images so the longest side is at most this (pixels)
WORKERS = 8                                     # questions sent to the server at the same time

EVALS = Path("../evals") if Path("../evals").is_dir() else Path("evals")
dev = [json.loads(line) for line in (EVALS / "dev.jsonl").read_text().splitlines()]

corpus = load_dataset("vidore/vidore_v3_pharmaceuticals", "corpus")["test"]
page_ids = corpus["corpus_id"]
pos_of = {cid: i for i, cid in enumerate(page_ids)}             # corpus_id -> row in corpus (to fetch its image)

retriever = bm25s.BM25()
retriever.index(bm25s.tokenize(corpus["markdown"], stopwords="english"))
positions, _ = retriever.retrieve(bm25s.tokenize([q["query"] for q in dev], stopwords="english"), k=TOP_PAGES)
for q, row in zip(dev, positions):
    q["shown"] = [page_ids[p] for p in row]                     # the pages the model will see

print(len(dev), "dev questions, first one sees pages", dev[0]["shown"])

/storage/homefs/kr23w045/vdoc-agent/envs/train/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
                                                                     

108 dev questions, first one sees pages [333, 331, 521]


In [2]:
client = OpenAI(base_url=BASE_URL, api_key="none")              # vLLM ignores the key
print([m.id for m in client.models.list()])                     # fails here if the server is not running

['Qwen/Qwen3-VL-4B-Instruct']


## 2. The answer call

Each page is sent as `Page <id>:` followed by its image, then the question.
`response_format` forces valid JSON, and the `enum` lets the model cite only the page ids it was shown.

In [8]:
SYSTEM = (
    "You answer questions about pharmaceutical documents using only the pages you are given. "
    "Each page is introduced by its page id. Answer in one or two sentences. "
    "In citations, list the ids of the pages that contain the answer. "
    "If the pages do not contain the answer, set abstain to true and leave answer empty."
)

def image_url(cid):
    """The page image as a base64 data URL (JPEG), resized to at most MAX_SIDE pixels."""
    img = corpus[pos_of[cid]]["image"].convert("RGB")
    img.thumbnail((MAX_SIDE, MAX_SIDE))
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=90)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

def answer(q):
    content = []
    for cid in q["shown"]:
        content.append({"type": "text", "text": f"Page {cid}:"})
        content.append({"type": "image_url", "image_url": {"url": image_url(cid)}})
    content.append({"type": "text", "text": f"Question: {q['query']}"})

    schema = {
        "type": "object",
        "properties": {
            "answer": {"type": "string"},
            "citations": {"type": "array", "items": {"type": "integer", "enum": q["shown"]}},
            "abstain": {"type": "boolean"},
        },
        "required": ["answer", "citations", "abstain"],
    }
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": content}],
        response_format={"type": "json_schema", "json_schema": {"name": "answer", "schema": schema}},
        temperature=0,
        max_tokens=300,
    )
    choice = resp.choices[0]
    try:
        out = json.loads(choice.message.content)
    except json.JSONDecodeError:
        # cut off before the JSON closed (finish_reason "length"), or otherwise broken: a failed answer
        out = {"answer": "", "citations": [], "abstain": False, "raw": choice.message.content}
    out["finish_reason"] = choice.finish_reason      
    # "stop" = finished normally, "length" = hit max_tokens
    return out


In [10]:
q = dev[0]
print("Q:  ", q["query"])
print("REF:", q["answer"])
print("correct pages:", [p["corpus_id"] for p in q["pages"]], "| shown:", q["shown"])
print(answer(q))

Q:   Does the Bolar exception help reduce the time it takes for generic drugs to reach the market after a patent expires?
REF: Yes, the Bolar exception helps reduce the time it takes for generic drugs to reach the market after a patent expires by allowing generic manufacturers to conduct the necessary studies and trials for regulatory approval while the original drug's patent is still active.
correct pages: [331, 332, 333, 335, 336, 337, 338, 339, 341, 342, 343, 344, 351, 521, 549, 560] | shown: [333, 331, 521]
{'answer': "Yes, the Bolar exception helps reduce the time it takes for generic drugs to reach the market after a patent expires by allowing third parties to use a patented invention for regulatory approval purposes without the patent holder's consent, which is necessary for preparing for market entry.", 'citations': [333], 'abstain': False, 'finish_reason': 'stop'}


## 3. Answer every dev question

In [11]:
with ThreadPoolExecutor(WORKERS) as pool:
    preds = list(tqdm(pool.map(answer, dev), total=len(dev)))

for q, p in zip(dev, preds):
    q["pred"] = p
print(Counter("abstain" if p["abstain"] else "answered" for p in preds))

  0%|          | 0/108 [00:00<?, ?it/s]

100%|██████████| 108/108 [01:23<00:00,  1.29it/s]

Counter({'answered': 99, 'abstain': 9})


In [12]:
print(Counter(p["finish_reason"] for p in preds))
for q, p in zip(dev, preds):
    if "raw" in p:
        print(q["query"], "\n   ", p["raw"][-300:], "\n")


Counter({'stop': 106, 'length': 2})
Explain how the FDA's research efforts on prescription drug promotion may influence the development of future regulatory policies. 
    bing and patient decisions, ultimately influencing regulatory policy. The research team's work is designed to ensure that promotional materials are effective and accurate, which can lead to more informed prescribing and patient decisions, ultimately influencing regulatory policy. The research team's 

Describe how the COVID-19 pandemic influenced the adoption of decentralized clinical trial models and their role in modernization efforts. 
    dated in 2021, reflects these lessons and emphasizes the importance of adapting to new technologies and processes. The pandemic has thus significantly influenced the modernization of clinical trials by promoting decentralized models and digital tools. The FDA's guidance on clinical trials during the 



## 4. Judge the answers

The reference answers are full sentences, so exact match does not work. A judge model reads question, reference and
prediction and returns `{"correct": true/false}`.

**v0 shortcut:** the judge is the same 4B model that answered. A model grading itself can be lenient; P12 replaces this
with a bigger judge checked against your own labels.

In [13]:
JUDGE = (
    "You grade an answer against a reference answer. The answer is correct if it states the same key facts as the "
    "reference: numbers, names and dates must match. Extra detail is fine. Missing or wrong key facts make it incorrect. "
    "An empty answer is incorrect."
)
JUDGE_SCHEMA = {"type": "object", "properties": {"correct": {"type": "boolean"}}, "required": ["correct"]}

def judge(q):
    prompt = f"Question: {q['query']}\nReference answer: {q['answer']}\nAnswer to grade: {q['pred']['answer']}"
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": JUDGE}, {"role": "user", "content": prompt}],
        response_format={"type": "json_schema", "json_schema": {"name": "grade", "schema": JUDGE_SCHEMA}},
        temperature=0,
        max_tokens=20,
    )
    return json.loads(resp.choices[0].message.content)["correct"]

with ThreadPoolExecutor(WORKERS) as pool:
    verdicts = list(tqdm(pool.map(judge, dev), total=len(dev)))
for q, v in zip(dev, verdicts):
    q["correct"] = v

100%|██████████| 108/108 [00:02<00:00, 48.25it/s]


## 5. Metrics

| metric | question it answers |
|---|---|
| accuracy | share of questions the judge marks correct |
| citation precision | of the pages the model cited, what share are correct pages? |
| citation recall | of the correct pages, what share did the model cite? |
| evidence shown | share of questions where at least one correct page was among the 3 shown |
| accuracy, evidence shown / not shown | separates **retrieval** failures (right page never shown) from **reading** failures |

In [14]:
def share(xs):
    xs = list(xs)
    return sum(xs) / len(xs) if xs else float("nan")

for q in dev:
    correct_pages = {p["corpus_id"] for p in q["pages"]}
    cited = set(q["pred"]["citations"])
    q["evidence_shown"] = bool(correct_pages & set(q["shown"]))
    q["cite_precision"] = len(cited & correct_pages) / len(cited) if cited else 0.0
    q["cite_recall"] = len(cited & correct_pages) / len(correct_pages)

metrics = {
    "accuracy": share(q["correct"] for q in dev),
    "citation_precision": share(q["cite_precision"] for q in dev),
    "citation_recall": share(q["cite_recall"] for q in dev),
    "abstain_rate": share(q["pred"]["abstain"] for q in dev),
    "evidence_shown": share(q["evidence_shown"] for q in dev),
    "accuracy_evidence_shown": share(q["correct"] for q in dev if q["evidence_shown"]),
    "accuracy_evidence_not_shown": share(q["correct"] for q in dev if not q["evidence_shown"]),
}
for m, v in metrics.items():
    print(f"{m:28} {v:.3f}")

accuracy                     0.500
citation_precision           0.603
citation_recall              0.371
abstain_rate                 0.083
evidence_shown               0.787
accuracy_evidence_shown      0.588
accuracy_evidence_not_shown  0.174


## 6. Read the failures

Wrong answers where the right page **was** shown: the model misread or the judge is wrong. Read them carefully.

In [15]:
reading_fails = [q for q in dev if q["evidence_shown"] and not q["correct"]]
print(len(reading_fails), "wrong answers although a correct page was shown\n")
for q in reading_fails[:5]:
    print("Q:   ", q["query"])
    print("REF: ", q["answer"])
    print("PRED:", q["pred"])
    print("content:", q["content_type"], "| correct pages:", [p["corpus_id"] for p in q["pages"]], "| shown:", q["shown"], "\n")

35 wrong answers although a correct page was shown

Q:    Analyze the role of international regulatory collaboration in addressing challenges within the generic drug industry.
REF:  International regulatory collaboration is critically relevant to addressing challenges in the generic drug industry by harmonizing standards, streamlining inspections, and enhancing global access to safe and affordable medicines.
PRED: {'answer': 'The document highlights international regulatory convergence as a key force shaping the future of the generic drug industry, implying that collaboration across borders is essential to address emerging challenges and stay ahead of global trends. However, the provided pages do not detail the specific mechanisms or outcomes of such collaboration.', 'citations': [917], 'abstain': False, 'finish_reason': 'stop'}
content: ['Text'] | correct pages: [916, 917, 918, 919] | shown: [1148, 710, 917] 

Q:    patterns in SEND usage growth for investigational new drug applicatio

## 7. Save predictions and log the run

In [16]:
from vdoc_agent.runlog import log_run

out = Path("../data/runs") if Path("../data").is_dir() else Path("data/runs")
out.mkdir(parents=True, exist_ok=True)
keep = ["query_id", "query", "answer", "content_type", "query_types", "shown", "pred", "correct",
        "evidence_shown", "cite_precision", "cite_recall"]
(out / "answers-v0-dev.jsonl").write_text("".join(json.dumps({k: q[k] for k in keep}) + "\n" for q in dev))

config = {"retriever": "bm25s", "top_pages": TOP_PAGES, "max_side": MAX_SIDE, "model": MODEL,
          "judge": MODEL, "split": "dev", "temperature": 0}
print(log_run("answer-v0", config, metrics))

/storage/homefs/kr23w045/vdoc-agent/reports/runs/20261007T013212Z_answer-v0.json
